# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

## ⚙️ Setup

In [10]:
%load_ext autoreload
%autoreload 0
%aimport _campaign_lib, api

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import json, os
from pprint import pprint
from api.models.pipeline_schema import PipelineSchema
from api.services.pipeline_discovery import TERMNORM_DEFAULT_SCHEMA
from _campaign_lib import *

In [12]:
svc = await init_services()
campaign_rounds = []
baseline_results = []

2026-03-09 13:14:21 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-09 13:14:21 INFO     [api.services.pipeline_discovery] Unknown pipeline ''; built schema from nodes dict with 6 steps
2026-03-09 13:14:21 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded:  vv1.1


Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93


## 🎛️ Configuration

In [13]:
campaign_config = {
    "queries_per_eval": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": 3,                 # default: 10
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "max_tokens": 4000,
    }
}

### Client info

In [17]:
# Fetch full pipeline config from backend (all parameters for reproducibility)
pipeline_raw = await svc["backend_client"].fetch_pipeline()
pipeline_config_full = pipeline_raw.get("data", pipeline_raw)
print(json.dumps(pipeline_config_full, indent=2))
print(f"{YELLOW}Nodes: {list(pipeline_config_full['nodes'])}")
print(f"{RESET }Nodes: {list(pipeline_config_full)}")
print(f"{GREEN }ID: {pipeline_config_full["name"]}  {pipeline_config_full["version"]}")

print(f"{RED}   PROMPTS: {RESET}{list(pipeline_config_full["resolved_schemas"])}")
print(f"{RED}SO-SCHEMAS: {RESET}{list(pipeline_config_full["resolved_prompts"])}")

2026-03-09 13:15:30 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


{
  "version": "v1.1",
  "available_models": [
    "meta-llama/llama-4-maverick-17b-128e-instruct",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b"
  ],
  "nodes": {
    "fuzzy_matching": {
      "type": "DeterministicFunction",
      "config": {
        "threshold": 70,
        "scorer": "WRatio",
        "limit": 5
      }
    },
    "web_search": {
      "type": "ExternalService",
      "config": {
        "max_sites": 7,
        "num_results": 20,
        "content_char_limit": 800,
        "url_fetch_multiplier": 2,
        "fallback_keywords_limit": 8,
        "query_prefix": "",
        "query_suffix": "",
        "brave_api_timeout": 10,
        "scrape_timeout": 5,
        "http_content_limit": 50000,
        "min_page_text_length": 200,
        "max_page_text_length": 10000,
        "title_truncate_length": 100,
        "scrape_workers": 10,
        "skip_extensions": [
          ".pdf",
          ".doc",
          

KeyError: 'name'

### Workflow Version

In [4]:
# Build active step params for evaluation (from synced experiment data)
pipeline_config = load_pipeline_config(svc["exp_data"])

# Which steps to EXCLUDE from evaluation (e.g. ["llm_ranking"] for token-matching-only)
EXCLUDE_STEPS = ["llm_ranking"]

pipeline_params = build_pipeline_params(
    pipeline_config,
    overrides=campaign_config.get("pipeline_overrides"),
    exclude_steps=EXCLUDE_STEPS,
)
campaign_config["pipeline_params"] = pipeline_params

print(f"\nActive steps: {pipeline_params['steps']}")



Active steps: ['entity_profiling', 'token_matching']


### 📊 Data

In [5]:
#@title Load datasets
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use stored data
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ────────────────────────────────────────────────
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


### Langfuse connection (optional)

In [6]:
#@title Langfuse — cloud sync config
# Credentials: set LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY in .env
LANGFUSE_PROJECT_NAME = "termnorm_ground_truth"
LANGFUSE_BACKFILL = True   # True = push all historical runs now
LANGFUSE_RESET = False      # True = clear push state first (re-push everything)

stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name=LANGFUSE_PROJECT_NAME,
    backfill=LANGFUSE_BACKFILL,
    reset=LANGFUSE_RESET,
)

  LANGFUSE PUSH (dataset-first)
Found 16 completed dataset runs for 'termnorm-local'
  Dataset 'termnorm_ground_truth': 15 items (0 created, 0 updated)

All 16 runs already pushed. Nothing to do.
  PUSH SUMMARY
  Total runs on disk:  16
  Newly pushed:        0
  Already done:        16
  Dataset:             termnorm_ground_truth
  Dataset items:       15


In [7]:
#@title Prepare evaluation context
baseline, eval_data, GROQ_API_KEY, backend_status = await prepare_eval_context(
    svc, train_data,
)

# OPTIONAL — uncomment to evaluate baseline prompt before search
# campaign_rounds, baseline_results = await run_baseline_eval(
#     baseline, eval_data, campaign_config, svc,
# )


BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   94
  Match Database Identifiers     109
  Match Database Aliases         593
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      meta-llama/llama-4-maverick-17b-128e-instruct
  ────────────────────────────────────────────────
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [8]:
#@title Candidate coverage (post-eval diagnostic)
cov_df = run_coverage_diagnostic(
    baseline_results,
    svc["store"], svc["backend_id"], svc["experiment_id"],
)

Loaded 40 eval queries
Eval runs: 16 completed runs, 3 in-progress
  run_id                name           model                      temp  accuracy  queries
  scan_15a5c1e9         scan                                      0.0   50.0%     6      
  scan_86f17bab         scan                                      0.0   66.7%     6      
  scan_fcb7bf9b         scan                                      0.0   50.0%     6      
  scan_01c3382c         scan                                      0.0   66.7%     6      
  scan_7d0c905a         scan                                      0.0   33.3%     6      
  scan_e9f03615         scan                                      0.0   66.7%     6      
  scan_3aff5881         scan                                      0.0   33.3%     6      
  scan_39b9fc27         scan                                      0.0   50.0%     6      
  scan_dff96710         scan                                      0.0   33.3%     6      
  scan_3dbc066d         scan     

## 🔍 Smart Search

In [9]:
#@title Task description (domain context for advisor)
TASK_DESCRIPTION_PATH = r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"

TASK_DESCRIPTION = load_task_description(TASK_DESCRIPTION_PATH)

Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md


In [10]:
#@title Scan advisor (run before sensitivity scan)

# Prefer live-enriched schema (has output_schema + prompt_meta); fall back to structural default
pipeline_schema = svc.get("pipeline_schema") or TERMNORM_DEFAULT_SCHEMA

# Filter variant library to active steps (respects EXCLUDE_STEPS)
_advisor_vl = load_variant_library()
if "pipeline_params" in campaign_config:
    from api.services.search.smart_search import filter_variant_library
    _advisor_vl = filter_variant_library(
        _advisor_vl, campaign_config["pipeline_params"], schema=pipeline_schema,
    )

# Ensure LLM client is ready
llm_client, llm_model = setup_llm(campaign_config, os.environ.get("GROQ_API_KEY", ""))

# Build coverage_stats from cov_df if available
_cov_stats = None
if "cov_df" in dir() and cov_df is not None and not cov_df.empty:
    _covered = int(cov_df["in_candidates"].sum())
    _total = len(cov_df)
    _cov_stats = {
        "covered": _covered,
        "total": _total,
        "coverage_pct": _covered / _total * 100 if _total else 0,
    }

advisory = await scan_advisor(
    pipeline_schema=pipeline_schema,
    variant_library=_advisor_vl,
    baseline_results=baseline_results,
    eval_data_size=len(train_data) if train_data else len(eval_data),
    llm_client=llm_client,
    model=llm_model,
    query_budget=120,
    coverage_stats=_cov_stats,
    excluded_steps=set(EXCLUDE_STEPS) if "EXCLUDE_STEPS" in dir() and EXCLUDE_STEPS else None,
    task_description=TASK_DESCRIPTION if "TASK_DESCRIPTION" in dir() else "",
)

# Extract pipeline_param axes proposed by the advisor (edit in next cell)
proposed_scan_variants, schema_labels = advisory_to_scan_variants(advisory, pipeline_schema=pipeline_schema)
print("\n--- PROPOSED SCAN VARIANTS (edit in next cell) ---")
for axis, values in proposed_scan_variants.items():
    if axis in schema_labels:
        print(f"  {axis}: (baseline + {len(values)-1} mutations)")
        for i, label in enumerate(schema_labels[axis]):
            print(f"    [{i}] {label}")
    else:
        print(f"  {axis}: {values}")

# Auto-populate from advisor output, or override manually:
if "proposed_scan_variants" in dir() and proposed_scan_variants:
    scan_variants = proposed_scan_variants
else:
    scan_variants = {}


2026-03-06 17:53:49 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR — pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Eval data size: 984 queries
  Query budget: 120
  Task context: # Domain Context: Life Cycle Assessment (LCA) Terminology

This document capture...
  Calling meta-llama/llama-4-maverick-17b-128e-instruct ...

──────────────────────────────────────────────────────────────────────
PRIORITY AXES (ranked by importance)
──────────────────────────────────────────────────────────────────────
  1. [HIGH] fuzzy_scorer (pipeline_param) — step: fuzzy_matching
     The fuzzy scorer directly affects the quality of candidate matches retrieved in the fuzzy matching step, which is critical for downstream steps and overall accuracy.
     Values: ['jaro-winkler', 'levenshtein', 'jaccard']
  2. [HIGH] fuzzy_threshold (pipeline_param) — step: fuzzy_matching
     The fuzzy threshold determines

In [11]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes use mutation tuples: ("-", path), ("+", path, type, required, desc),
# ("~", old_path, new_name, type, required, desc). Dot notation for nesting.
# Non-schema axes use plain value lists.

# Override / edit here:
scan_variants = {
    'max_token_candidates': [15, 20, 25],
    'profiling_temperature': [0.0, 0.3, 0.7],
    'profiling_schema': [
        [('-', 'notes'), ('-', 'material_classification')],
        [('+', 'significance', 'string', True, 'Domain relevance of entity')],
        [('~', 'notes', 'industry_sector', 'string', True, 'Industry sector classification')],
    ],
    'relevance_weight_core': [0.5, 0.7, 0.9],
}

# Resolve schema mutation tuples into concrete JSON Schema dicts
from api.models.schema_mutation import (
    parse_mutation_tuples, resolve_schema_variants, baseline_schema_from_step,
)
from api.services.backend_client import FLAT_TO_NODE_OVERRIDE

_resolved_variants = {}
_schema_labels = {}
for _axis, _vals in scan_variants.items():
    if _axis.endswith("_schema") and _vals and isinstance(_vals[0], list):
        _oi = FLAT_TO_NODE_OVERRIDE.get(_axis)
        _step = pipeline_schema.get_step(_oi[0]) if _oi else None
        if _step and _step.output_schema:
            _parsed = [parse_mutation_tuples(v) for v in _vals]
            _bl = baseline_schema_from_step(_step.output_schema)
            _resolved_variants[_axis] = resolve_schema_variants(_bl, _parsed)
            _schema_labels[_axis] = ["(baseline)"] + [v.render_label() for v in _parsed]
            continue
    _resolved_variants[_axis] = _vals

scan_variants = _resolved_variants

# Display
for _axis, _vals in scan_variants.items():
    if _axis in _schema_labels:
        print(f"  {_axis}: (baseline + {len(_vals)-1} mutations)")
        for i, label in enumerate(_schema_labels[_axis]):
            print(f"    [{i}] {label}")
    else:
        print(f"  {_axis}: {_vals}")


2026-03-06 17:53:52 WARNING  [api.models.schema_mutation] Remove target 'material_classification' not found in properties; skipping


  max_token_candidates: [15, 20, 25]
  profiling_temperature: [0.0, 0.3, 0.7]
  profiling_schema: (baseline + 3 mutations)
    [0] (baseline)
    [1] ('-', 'notes'), ('-', 'material_classification')
    [2] ('+', 'significance', 'string', True, 'Domain relevance of entity')
    [3] ('~', 'notes', 'industry_sector', 'string', True, 'Industry sector classification')
  relevance_weight_core: [0.5, 0.7, 0.9]


In [12]:
#@title 🔬 Smoke test: node_overrides on /matches
# await smoke_test_override(svc, pipeline_config_full, eval_data)

In [13]:
#@title Build diagnostic set (with resume)
(plan_id, search_baseline, diagnostic,
 cached_profiles, variant_library) = await resume_or_build_diagnostic(
    campaign_config, baseline, baseline_results,
    svc, eval_data,
    scan_variants=scan_variants,
)

2026-03-06 17:53:53 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)
2026-03-06 17:53:53 INFO     [api.services.search.smart_search] Scan complete in plan ssplan_5ce4991c339d, reusing profiles


[RESUME] Plan ssplan_5ce4991c339d: 4 axis profiles available


In [14]:
#@title Historical data audit & inventory
prompt_index, cached_profiles = audit_historical_data(
    svc["store"], svc["backend_id"],
    diagnostic, cached_profiles,
)

2026-03-06 17:53:53 INFO     [api.services.search.coverage] build_prompt_result_index: 16 runs -> 3 unique prompts, 36 total query results


  DATA INVENTORY  (3 prompts, 36 query results)
  Baselines: 1 plan baseline(s) — 6 queries cached

  No axis variations found in stored plans.

  Pipeline parameters (from sensitivity scans):
    profiling_schema         4 values scanned  sensitivity: 0.300  [medium]
    max_token_candidates     3 values scanned  sensitivity: 0.150  [skip]
    profiling_temperature    3 values scanned  sensitivity: 0.150  [skip]
    relevance_weight_core    3 values scanned  sensitivity: 0.000  [skip]

  Identified: 1/3 prompts (6/36 queries) via stored plans
  Unmatched:  2 prompts (30 queries)


In [15]:
#@title Coverage advisor
# Knobs: adjust these and re-run to see different strategies
min_queries = 6          # min queries per variant to count as "usable"
axis_requirements = None  # None = require all values; or e.g. {"persona": 2}

coverage = show_scan_coverage(
    search_baseline, variant_library, diagnostic,
    prompt_index,
    pipeline_params=campaign_config.get("pipeline_params"),
    min_queries=min_queries,
    axis_requirements=axis_requirements,
)

  COVERAGE ADVISOR  (min_queries=6)
  Baseline: 6/6 queries cached ✓

  Pipeline params (always need backend):
    max_token_candidates   3 variants × 6 queries = 18 calls
    profiling_temperature  3 variants × 6 queries = 18 calls
    profiling_schema       4 variants × 6 queries = 24 calls
    relevance_weight_core  3 variants × 6 queries = 18 calls

  Summary: 0 cached, 78 still needed (78 pipeline-param)
  >> 0/0 prompt field axes covered. Run scan to fill gaps on: max_token_candidates, profiling_temperature, profiling_schema, relevance_weight_core. Tip: lower min_queries or reduce axis_requirements to accept sparser coverage.

  Note: 36 results exist in the index under other baselines.
  The current search_baseline was rebuilt and has no cached data yet.


In [16]:
#@title Sensitivity scan
scan_df = None
axis_profiles = []

# Detect plan status for resume
_plan_data = svc["store"].smart_search.load(svc["backend_id"], plan_id)
_plan_status = _plan_data.get("status", "") if _plan_data else ""

# Load partial checkpoint if scan was interrupted
_partial_scan = None
if _plan_data and _plan_status == "scan_partial":
    _sr = _plan_data.get("scan_results", {})
    _partial_scan = {
        "rows": _sr.get("rows", []),
        "completed_axes": _sr.get("completed_axes", []),
    }
    print(f"[RESUME] Found partial scan: {len(_partial_scan['completed_axes'])} axes done")

scan_df, axis_profiles = await sensitivity_scan(
    search_baseline, variant_library, diagnostic, svc.get("backend_client"),
    user_focus=campaign_config.get("improvement_areas", ""),
    store=svc["store"], backend_id=svc["backend_id"],
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
    plan_id=plan_id,
    prompt_result_index=prompt_index,
    partial_scan=_partial_scan,
    pipeline_schema=svc.get("pipeline_schema"),
)

2026-03-06 17:53:53 INFO     [api.services.search.coverage] build_prompt_result_index: 16 runs -> 3 unique prompts, 36 total query results


Running sensitivity scan...
  User focus: profile schema quality, web search relevance

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Analyze entity profile and evaluate candidate matches based on core concept and ...
    problem_description: Evaluating candidates against a given entity profile and core concept.
    instruction: First, analyze the provided ENTITY PROFILE (JSON): {{entity_profile_json}} to id...
    thinking_style: Think step-by-step: Identify key features, evaluate candidates, and rank them ba...
    answer_format: {"reasoning": "[1-2 sentences explaining profile, entity_category, and key featu...

  Estimated configs: ~13 x 6 queries
  Evaluating baseline...
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
        MISS 2/20  [token]  SJRG0010-ABS/mold

2026-03-06 17:53:53 INFO     [api.services.search.smart_search] Checkpoint: axis 'max_token_candidates' complete (1/4)


  [2] 25                                         4/6  66.7%  +15.0% ^ [cached]

  Axis 2/4: profiling_schema (pipeline_param, 4 values)
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
        MISS 2/20  [token]  SJRG0010-ABS/molding                           -> Extrusion, plastic pipes {RER}| ext ⚡
        MISS 2/20  [token]  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 60 ⚡
        MISS 11/20  [token]  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 ⚡
        MISS           PA6/66 Ultramid C3U/molding                    ERR: Client error '400 Bad Request' for url ' ⚡
  [0] schema(11 fields)                          2/6  33.3%  -15.0% v [cached]
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced 

2026-03-06 17:53:53 INFO     [api.services.search.smart_search] Checkpoint: axis 'profiling_schema' complete (2/4)


...

  [2] 0.7                                        3/6  50.0%  +0.0% [cached]

  Axis 4/4: relevance_weight_core (pipeline_param, 3 values)
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
        MISS 3/20  [token]  SJRG0010-ABS/molding                           -> Acrylonitrile-butadiene-styrene cop ⚡
        HIT   [token]  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 75 ⚡
        MISS 11/20  [token]  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 ⚡
        MISS 3/20  [token]  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 ⚡
  [0] 0.5                                        3/6  50.0%  +0.0% [cached]
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
 

2026-03-06 17:53:54 INFO     [api.services.search.smart_search] Checkpoint: axis 'relevance_weight_core' complete (4/4)


  [2] 0.9                                        3/6  50.0%  +0.0% [cached]
  >> profiling_schema: range=30.0%, best=+15.0%, worst=-15.0%, budget=medium
  >> max_token_candidates: range=15.0%, best=+15.0%, worst=+0.0%, budget=skip
  >> profiling_temperature: range=15.0%, best=+0.0%, worst=-15.0%, budget=skip
  >> relevance_weight_core: range=0.0%, best=+0.0%, worst=+0.0%, budget=skip


2026-03-06 17:53:54 INFO     [api.services.search.smart_search] Saved scan results to plan: ssplan_5ce4991c339d



Sensitivity scan complete: 13 variants evaluated

Rank  Axis                      Type            Card  Range    Budget  
----------------------------------------------------------------------
  1   profiling_schema          pipeline_param  4     0.300    medium  
  2   max_token_candidates      pipeline_param  3     0.150    skip    
  3   profiling_temperature     pipeline_param  3     0.150    skip    
  4   relevance_weight_core     pipeline_param  3     0.000    skip    


In [17]:
#@title Select best from scan & seed campaign
best_ps, best_params = select_scan_winner_notebook(
    scan_df, axis_profiles, search_baseline, variant_library,
    pipeline_params=campaign_config.get("pipeline_params"),
    store=svc["store"], backend_id=svc["backend_id"], plan_id=plan_id,
)

if best_params:
    campaign_config["pipeline_params"] = best_params
    print(f"Updated pipeline_params: {best_params}")

campaign_rounds.append({
    "round": "search",
    "label": f"smart_search ({best_ps.changes_description or best_ps.id[:12]})",
    "prompt_state": best_ps,
    "accuracy": campaign_rounds[0]["accuracy"] if campaign_rounds else 0.0,
    "hits": campaign_rounds[0].get("hits", 0) if campaign_rounds else 0,
    "total": campaign_rounds[0].get("total", 0) if campaign_rounds else 0,
    "results": campaign_rounds[0].get("results", []) if campaign_rounds else [],
})
display_progress(campaign_rounds)

2026-03-06 17:53:54 INFO     [api.services.search.smart_search] select_scan_winner: 0 prompt changes, 1 param changes from 1 improving axes


Selected best from 1 improving axes:
  profiling_schema          best_delta=+15.0%  value_idx=1  acc=66.7%
Pipeline params updated: {'steps': ['entity_profiling', 'token_matching'], 'profiling_schema': {'type': 'object', 'properties': {'entity_name': {'type': 'string'}, 'core_concept': {'type': 'string', 'description': 'The single word that defines what this expression represents'}, 'distinguishing_features': {'type': 'array', 'items': {'type': 'string'}}, 'key_properties': {'type': 'array', 'items': {'type': 'string'}}, 'technical_specifications': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Explicit technical specs, dimensions, codes, ratings, tolerances'}, 'alternative_names': {'type': 'array', 'items': {'type': 'string'}}, 'classification_aliases': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Full spectrum of valid ways this entity could be referenced using expert-level terminology, from precise to generic'}, 'constituent_materials': {'type': 'ar

## 🗺️ Grid Search (Optional)

<details>
<summary>Skip if you used Smart Search above. Expand for brute-force grid sweep.</summary>

**What:** Systematic sweep of the prompt configuration space (Layer 1 fields) using a cartesian product of default axis variations. Maps the accuracy landscape before hill-climbing.

**When to use:** When you want exhaustive coverage of the grid, or when Smart Search results look unreliable and you want independent validation.

**What you get:** Ranked starting points, which dimensions matter most (marginal stats), interaction effects between fields (heatmaps), and LLM-analyzed insights.

**How to read results:**
- **Ranked table** — best combos at the top; use the winner as your campaign seed
- **Marginal stats** — which axis values have the highest mean accuracy across all combos
- **Pairwise heatmaps** — green = good interaction, red = bad; look for synergies and conflicts
- **LLM analysis** — automated pattern recognition across the grid results

</details>

In [ ]:
#@title Grid campaign overview (existing plans)
campaign_config["grid_search"] = {
    "context": "A terminology normalization pipeline that matches raw material "
                "descriptions to standardized database terms using entity profiling "
                "and candidate ranking.",
    "grid_budget": 35,               # default: 0 (full grid)
    "eval_queries_per_point": 6,     # default: 0 (all queries)
    "shared_queries": False,         # default: True
}


merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
#@title Build or resume grid plan + load eval data
gs = campaign_config["grid_search"]
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=campaign_config.get("improvement_areas", ""),
)

# Load full eval_data for later optimization rounds
eval_data = load_eval_dataset(
    svc["store"], svc["backend_id"], svc["experiment_id"],
)
if not eval_data:
    raise RuntimeError(
        "No evaluation data found. Generate data first "
        "(e.g. run evaluation.ipynb or load from DatasetStore)."
    )

print(f"Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

In [ ]:
#@title Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"],
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

In [ ]:
#@title Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

In [ ]:
#@title Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

## 🚀 Optimization

<details>
<summary>Details</summary>

Two modes:
- **Semi-automatic** (recommended): runs multiple rounds with patience-based auto-stop
- **Manual**: run one round at a time for full HITL control

Both modes subsample `eval_data` to `queries_per_eval` queries per step.

</details>

In [18]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
)


  FEEDBACK CYCLE - PRE-FLIGHT
  Baseline accuracy      : 0.0%
  Baseline prompt        : First, analyze the provided ENTITY PROFILE (JSON): {{entity_profile_json}} to id...
  ------------------------------------------------------------------
  Max rounds             : 3
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : disabled
  L3 (modify plan)       : disabled
  ------------------------------------------------------------------
  Candidate model        : meta-llama/llama-4-maverick-17b-128e-instruct
  Creativity             : 0.7
  Active steps           : entity_profiling, token_matching
  ------------------------------------------------------------------
  Est. backend calls     : 225  (3 rounds x 5 cands x 15 queries)


2026-03-06 17:54:00 INFO     [api.services.campaign.feedback_cycle] Using provided baseline (acc=0.000)
2026-03-06 17:54:00 INFO     [api.services.campaign.feedback_cycle] Cycle identity: cycle_cc780ebd7ba2
2026-03-06 17:54:00 INFO     [api.services.campaign.feedback_cycle] Cycle cycle_cc780ebd7ba2 marked completed but has no trials — will restart
2026-03-06 17:54:00 INFO     [api.services.campaign.feedback_cycle] Cycle cycle_cc780ebd7ba2 exists but has no completed rounds — starting fresh
2026-03-06 17:54:01 WARNING  [api.services.obs.observability_logger] Skipping Langfuse cloud dataset registration for 984 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-03-06 17:54:01 INFO     [api.services.campaign.feedback_cycle] Registered 728 dataset items for 'termnorm_ground_truth'
2026-03-06 17:54:01 INFO     [api.services.campaign.feedback_cycle] Feedback cycle round 0 (acc=0.000, stall=0/2)
2026-03-06 17:54:01 INFO     [api.services.campaign.feedback_cycle] Loade

  [1] C1/5 Q1/15
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
  [2] C1/5 Q2/15
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
  [3] C1/5 Q3/15
        HIT   [token]  SJRG0010-ABS/molding                           -> Injection moulding {RER}| injection ⚡
  [4] C1/5 Q4/15
        HIT   [token]  Kingfa NPG25                                   -> Glass fibre reinforced plastic | 75 ⚡
  [5] C1/5 Q5/15
        MISS 10/20  [token]  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 ⚡
  [6] C1/5 Q6/15
        MISS 3/20  [token]  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 ⚡
  [7] C1/5 Q7/15
        MISS 9/20  [token]  PC  GF10  makrolon material/0                  -> Glass fibre reinforced plastic | 90 ⚡
  [8] C1/5 Q8/15
        MISS 2/20  [token]  Copper Wire/cold forming                       -> M

2026-03-06 17:54:01 INFO     [api.services.prompt_eval] Resuming candidate_2_bc44210c: 4 cached results, 11 remaining


  >> Candidate 2/5 done: 26.7% (c=0.2400)
  [31] C3/5 Q1/15
        HIT   [token]  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 75 ⚡
  [32] C3/5 Q2/15
        HIT   [token]  Stainless steel EN 10270-3/winding             -> Wire drawing, steel {RER}| wire dra ⚡
  [33] C3/5 Q3/15
        HIT   [token]  SJRG0010-ABS/molding                           -> Injection moulding {RER}| injection ⚡
  [34] C3/5 Q4/15
        MISS 2/20  [token]  Kingfa NPG25                                   -> Polyamide (Nylon) 6.6/EU-27 ⚡
  [35] C3/5 Q5/15
        MISS 11/20  [token]  PA 66 25% GF V0 RAL 7012/0                     -> Glass fibre reinforced plastic | 75 1.7s
  [36] C3/5 Q6/15
        MISS 2/20  [token]  PA6/66 Ultramid C3U/molding                    -> Polyamide (Nylon) 6.6/EU-27 1.9s
  [37] C3/5 Q7/15
        MISS 11/20  [token]  PC  GF10  makrolon material/0                  -> Glass fibre reinforced plastic | 90 1.6s
  [38] C3/5 Q8/15
        MISS 4/20  [to

2026-03-06 17:54:24 WARNING  [api.services.prompt_eval] Eval batch interrupted at query 10/11. Partial results saved via incremental writer.
2026-03-06 17:54:24 WARNING  [api.services.prompt_eval] evaluate_prompt_cached interrupted for candidate_2_bc44210c. Partial results saved to candidate_2_bc44210c.partial.jsonl — will resume on next run.
2026-03-06 17:54:24 WARNING  [api.services.campaign.feedback_cycle] Feedback cycle interrupted at round 0. Completed rounds are checkpointed.



OPTIMIZATION COMPLETE (feedback cycle)
  Rounds run: 0
  Best accuracy: 0.0% (round search)
  Stop reason: interrupted
  Cycle ID: cycle_cc780ebd7ba2
  Langfuse trace: 605f8795e3eacc00fde3bb61e9128493


In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 💡 LLM Suggestions

<details>
<summary>Details</summary>

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 5-6.**

</details>

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print(f"--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 📋 Results

<details>
<summary>Details</summary>

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

</details>

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    if not base_r or not final_r:
        print("Skipping flip tracking — baseline or final results are empty.")
    else:
        flips = []
        for br, fr in zip(base_r, final_r):
            b_hit = br["hit"]
            f_hit = fr["hit"]
            if b_hit != f_hit:
                flips.append({
                    "query": br["query"][:50],
                    "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                    "base_pred": br["predicted"][:35],
                    "final_pred": fr["predicted"][:35],
                    "ground_truth": br["ground_truth"][:35],
                })

        gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
        lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

        print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
        print(f"  Queries gained (MISS->HIT): {gained}")
        print(f"  Queries lost (HIT->MISS):   {lost}")
        print(f"  Net change:                 {gained - lost:+d}")
        print()
        if flips:
            display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name=LANGFUSE_PROJECT_NAME,
)